# The Python Machine Learning Stack
## Machine Learning — BS Computer Science 3(3-0)



| | |
|---|---|
| **Notebook** | `NB02_Python_ML_Stack.ipynb` |
| **Lecture** | Lecture 3 (Week 1) |
| **CLO** | CLO-1 / CLO-5 |
| **Dataset** | Iris, plus a real CSV of your choice |

**Objective.** Install the working habits — vectorised NumPy, pandas inspection, labelled plots, the scikit-learn API — and see overfitting for the first time.

**By the end of this notebook you will be able to:**

1. Use vectorised NumPy operations and explain the axis argument.
2. Perform the six-step first look at any dataset with pandas.
3. Produce fully labelled histograms and class-coloured scatter plots.
4. Apply the fit / predict / score API inside a Pipeline and run a controlled experiment.

---

## 0. Setup and reproducibility

In [2]:
# ---------------------------------------------------------------------
# Reproducibility header — required in EVERY notebook you submit
# ---------------------------------------------------------------------
import sys, numpy as np, pandas as pd, sklearn, matplotlib
import matplotlib.pyplot as plt

print('python      ', sys.version.split()[0])
print('numpy       ', np.__version__)
print('pandas      ', pd.__version__)
print('scikit-learn', sklearn.__version__)
print('matplotlib  ', matplotlib.__version__)

RANDOM_STATE = 1
rng = np.random.default_rng(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

ModuleNotFoundError: No module named 'matplotlib'

## 1. NumPy — vectorise, never loop

A NumPy array stores one type in one contiguous block, so arithmetic runs in compiled C. **`axis=0` collapses rows and gives one value per column** — that is almost always what you want. Mixing up the axis is the most common NumPy bug in student code.

In [ ]:
a = np.array([[1., 2., 3.],
              [4., 5., 6.]])          # shape (2, 3)

print('shape       ', a.shape, ' dtype', a.dtype)
print('mean axis=0 ', a.mean(axis=0), '  <- per COLUMN (one per feature)')
print('mean axis=1 ', a.mean(axis=1), '        <- per ROW')
print('a * 2       ', a[0] * 2)
print('column 1    ', a[:, 1])
print('mask a > 3  ', a[a > 3])

z = (a - a.mean(axis=0)) / a.std(axis=0)      # standardisation
print('standardised:\n', np.round(z, 3))

### Exercise 1.1 — loop versus vectorised

Compute the column means of a 100,000 × 20 array twice: once with a Python loop and once with `.mean(axis=0)`. Time both with `%%timeit` and report the ratio.

In [ ]:
big = rng.normal(size=(100_000, 20))

# TODO: time a Python loop that computes the column means
# TODO: time big.mean(axis=0)
# TODO: print the speed ratio and write one sentence about it

In [ ]:
big = rng.normal(size=(100_000, 20))

# Python loop
%timeit -n 5 -r 3 [
    sum(big[i, j] for i in range(big.shape[0])) / big.shape[0]
    for j in range(big.shape[1])
]

# NumPy vectorized operation
%timeit -n 5 -r 3 big.mean(axis=0)

# Speed ratio
import time

start = time.perf_counter()
loop_means = [
    sum(big[i, j] for i in range(big.shape[0])) / big.shape[0]
    for j in range(big.shape[1])
]
loop_time = time.perf_counter() - start

start = time.perf_counter()
vector_means = big.mean(axis=0)
vector_time = time.perf_counter() - start

print("Speed ratio:", loop_time / vector_time)
print("NumPy vectorization is much faster than the Python loop because the operation is executed using optimized compiled code.")

IndentationError: unexpected indent (1539172296.py, line 5)

## 2. pandas — the six-step first look

Run these on a **real, messy CSV**, not on Iris. What you are hunting for: shape, missing values, impossible values, class balance and feature scales.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target

print(df.shape)                                   # 1. big enough?
display(df.head())                                # 2. always look
df.info()                                         # 3. dtypes, non-null
display(df.describe().T[['min', 'max', 'mean', 'std']].round(2))   # 4. impossible values?
print(df['target'].value_counts(normalize=True))  # 5. class balance
print(df.isnull().sum().sum(), 'missing values')  # 6. missingness

**Interpretation.** Look at the `min`/`max` column of `describe()`. Which two features have the most different scales, and by what factor? Which models from Week 3 onwards would that break?

The two features with the most different scales are proline and alcohol. Proline ranges from about 278 to 1680, while alcohol ranges from about 11 to 15. Proline therefore has a much larger numerical scale. Distance-based models such as k-NN are strongly affected by this difference because features with larger values can dominate the distance calculation. Scaling is therefore important for k-NN.

## 3. Matplotlib — the four plots you will draw all semester

Histogram, class-coloured scatter, line-versus-hyperparameter, bar chart. **Every plot needs axis labels with units and a legend.** An unlabelled plot carries no information and earns no marks.

In [ ]:
X, y = wine.data, wine.target

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

ax[0].hist(X[:, 0], bins=25, edgecolor='black')
ax[0].set_xlabel('alcohol [% vol]')
ax[0].set_ylabel('count')
ax[0].set_title('Histogram — distribution of one feature')

for cl, m in zip(np.unique(y), ('o', 's', '^')):
    ax[1].scatter(X[y == cl, 0], X[y == cl, 9], marker=m,
                  alpha=0.8, edgecolor='k', label=f'class {cl}')
ax[1].set_xlabel('alcohol [% vol]')
ax[1].set_ylabel('colour intensity')
ax[1].set_title('Scatter — are the classes separable?')
ax[1].legend()

plt.tight_layout(); plt.show()

## 4. Your first complete machine-learning program

Eleven lines. Every one of them corresponds to a concept from Lecture 2.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

pipe = Pipeline([('scaler', StandardScaler()),
                 ('knn', KNeighborsClassifier(n_neighbors=5))])
pipe.fit(X_train, y_train)

print('train accuracy %.3f' % pipe.score(X_train, y_train))
print('test  accuracy %.3f' % pipe.score(X_test,  y_test))

> **The most important line on this page.** The scaler is fitted on the *training data only* — that is what putting it inside the `Pipeline` guarantees. Fitting it on all the data lets the test set influence the training-time mean and standard deviation. That is **data leakage** and Week 8 is devoted to it.

## 5. Controlled experiment — change one thing at a time

Record the results in the table **before** you interpret anything. This is the standard experimental protocol for the rest of the course.

In [ ]:
rows = []
for k in (1, 5, 25, 100):
    p = Pipeline([('scaler', StandardScaler()),
                  ('knn', KNeighborsClassifier(n_neighbors=k))])
    p.fit(X_train, y_train)
    rows.append({'variant': 'scaled',   'k': k,
                 'train': p.score(X_train, y_train),
                 'test':  p.score(X_test,  y_test)})

    raw = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    rows.append({'variant': 'unscaled', 'k': k,
                 'train': raw.score(X_train, y_train),
                 'test':  raw.score(X_test,  y_test)})

results = pd.DataFrame(rows).pivot(index='k', columns='variant',
                                   values=['train', 'test']).round(3)
display(results)

### Interpretation — three things to explain

1. At `k = 1` the **training** accuracy is exactly 1.000. Why is that guaranteed, and what does the number tell you about the model?
2. What happened when you removed the scaler, and why does k-NN care about units?
3. What happens to the test accuracy as `k` approaches the size of the training set?

1. At k = 1, the training accuracy is exactly 1.000 because each training sample is its own nearest neighbor. Therefore, the model can correctly classify every training sample. This indicates that k = 1 can fit the training data very closely and may overfit.

2. When the scaler was removed, the performance of k-NN generally decreased. k-NN uses distances between data points, so features with larger numerical values can have a greater effect on the distance. Scaling puts the features on comparable scales.

3. As k approaches the size of the training set, the model considers a very large number of training samples when making a prediction. The predictions become less sensitive to individual samples and the model becomes more generalized, but it can become too simple and underfit the data.

---
## Deliverable

The timing exercise (1.1), the six-step first look on a real CSV, two fully labelled plots, and the four-row results table with all three interpretations written out.

---
## Submission checklist

Tick every box before you submit. Items 1–6 are the course reproducibility
rules from Lecture 3 and are marked in the rubric.

- [ ] The random seed is fixed **and printed**
- [ ] Library versions are printed in the first cell
- [ ] The raw data file was **not** modified
- [ ] Preprocessing happens **after** the split, inside a `Pipeline`
- [ ] The dataset source and licence are stated in a markdown cell
- [ ] **Kernel → Restart & Run All** completes without error
- [ ] Every experiment is followed by a written interpretation
- [ ] At least one wrong prediction or failure case is explained
- [ ] Results are reported as **mean ± standard deviation** where cross-validated
- [ ] Every plot has axis labels with units and a legend
